In [ ]:
import math
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.gridspec as gridspec
import pandas as pd

from os.path import join
import os
from functools import partial
import pathlib
from pathlib import Path
import shutil

from joblib import Parallel, delayed
import joblib

import gc
import subprocess

from credit.pbs import get_num_cpus

In [ ]:
# parameters
forecast_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06"
c13_only = 0
num_cpus = 2
test = 0
config = None
delay = 50
dpi = 300
max_steps = 12

In [ ]:
if c13_only:
    channels = [13]
else:
    channels = [7, 8, 9, 10, 13]

In [ ]:
def plot_ensemble_diff_forecast(ds, true_ds, init_ds, forecast_dir, channels, img_idx=-1, diff_max=25):
    true_ds["t"] = ds.t

    for channel in channels:
        # Extract true and squeeze
        true = true_ds.sel(channel=channel).BT_or_R.squeeze()

        # Extract prediction (Keep ensemble dim, squeeze others)
        preds = ds.sel(channel=channel).BT_or_R.squeeze()

        if "ensemble_member_label" in preds.dims:
            num_members = preds.ensemble_member_label.size
            member_labels = preds.ensemble_member_label.values
        else:
            num_members = 1
            member_labels = [0]
            
        # Calculate differences for all members
        diffs = preds - true

        # 3 plots per member: Obs, Pred, Diff
        cols = 12 # Increased to 12 columns
        members_per_row = cols // 3 # Now fits 4 members per row
        rows = math.ceil(num_members / members_per_row)
        total_plots = rows * cols

        # Create dynamic grid (Increased width to 32 to fit 12 columns)
        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(32, 3.5 * rows), 
            subplot_kw={"projection": ccrs.PlateCarree()},
        )
        axes = np.atleast_1d(axes).flatten()

        # Colormap settings based on channel
        if channel > 4:
            cmap_raw, vmin_raw, vmax_raw = "Spectral_r", 200, 300
            vmax_diff = diff_max
        else:
            cmap_raw, vmin_raw, vmax_raw = "Spectral_r", 0.0, 0.2
            vmax_diff = 0.1

        # Plot loop
        for i in range(num_members):
            # Calculate the exact axes indices for this member's triplet
            row_idx = i // members_per_row
            col_group = i % members_per_row
            
            idx_obs = row_idx * cols + col_group * 3
            idx_pred = idx_obs + 1
            idx_diff = idx_obs + 2
            
            if "ensemble_member_label" in preds.dims:
                member_pred = preds.isel(ensemble_member_label=i)
                member_diff = diffs.isel(ensemble_member_label=i)
                title_pred = f"Ens {member_labels[i]}"
            else:
                member_pred = preds
                member_diff = diffs
                title_pred = "Prediction"

            # 1. Plot Truth
            ax_obs = axes[idx_obs]
            im_raw = true.plot(ax=ax_obs, transform=ccrs.PlateCarree(), cmap=cmap_raw, vmax=vmax_raw, vmin=vmin_raw, add_colorbar=False)
            ax_obs.set_title("Observations")

            # 2. Plot Prediction
            ax_pred = axes[idx_pred]
            member_pred.plot(ax=ax_pred, transform=ccrs.PlateCarree(), cmap=cmap_raw, vmax=vmax_raw, vmin=vmin_raw, add_colorbar=False)
            ax_pred.set_title(title_pred)
            
            # 3. Plot Difference
            ax_diff = axes[idx_diff]
            im_diff = member_diff.plot(ax=ax_diff, transform=ccrs.PlateCarree(), cmap="seismic", vmax=vmax_diff, vmin=-vmax_diff, add_colorbar=False)
            ax_diff.set_title(f"{title_pred} - Obs")

        # Formatting axes
        for i in range(num_members * 3):
            ax = axes[i]
            ax.add_feature(cfeature.COASTLINE)
            ax.set_xticks(list(range(-120, -29, 30)))
            ax.set_yticks(list(range(-50, 51, 25)))

            # Only show y-labels on the far left column
            if i % cols == 0:
                ax.set_ylabel("Latitude", fontsize=10)
            else:
                ax.set_ylabel("")
                ax.tick_params(labelleft=False)

            # Only show x-labels on the bottom row
            if i >= (rows - 1) * cols:
                ax.set_xlabel("Longitude", fontsize=10)
            else:
                ax.set_xlabel("")
                ax.tick_params(labelbottom=False)

        # Hide completely unused axes at the bottom right
        for i in range(num_members * 3, total_plots):
            axes[i].axis("off")

        # Two Shared Colorbars at the bottom
        cbar_y = 0.05 / rows
        cbar_ax_raw = fig.add_axes([0.25, cbar_y, 0.20, 0.02])
        cbar_ax_diff = fig.add_axes([0.55, cbar_y, 0.20, 0.02])
        
        fig.colorbar(im_raw, cax=cbar_ax_raw, orientation="horizontal", label="BT or R")
        fig.colorbar(im_diff, cax=cbar_ax_diff, orientation="horizontal", label="Difference")

        # Title & Save
        time_val = preds.t.values
        if hasattr(time_val, "item"):
            time_val = time_val.item()
        time_str_title = pd.Timestamp(time_val).strftime("%Y-%m-%dT%H:%M:%S")

        fig.subplots_adjust(top=0.95, bottom=cbar_y + (0.30 / rows), hspace=0.075, wspace=0.1)
        fig.suptitle(f"Channel {channel}, forecast step {img_idx:02}\n{time_str_title}", y=0.98, fontsize=16)

        figname = f"C{channel:02}_step{img_idx:02}.png"
        path1 = pathlib.Path(forecast_dir)
        time_str = path1.parent.name if not path1.is_dir() else path1.name
        
        save_dir = join(path1.parent, "ensemble_diff_gif", f"gifs_{time_str}", f"C{channel:02}")
        os.makedirs(save_dir, exist_ok=True)

        plt.savefig(join(save_dir, figname), format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)


In [ ]:
num_cpus = max(1, get_num_cpus() - 1)
print(f"using {num_cpus} cpus")
print(f"processing {forecast_dir}")

files = sorted(
    [f for f in os.listdir(forecast_dir) if os.path.isfile(join(forecast_dir, f))]
)[:max_steps]

if test:
    print("WARNING: test mode!")
    files = files[:2]
    num_cpus = 2

def make_gif(forecast_dir, channels, file_and_index):
    k, file = file_and_index
    img_idx = k + 1

    zarr_ds = (
        xr.open_dataset(
            "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr",
            consolidated=False,
        )
        .drop_duplicates(dim="t")
        .sortby("t")
        .transpose("t", "channel", "latitude", "longitude")
    )

    path1 = pathlib.Path(forecast_dir)
    time_str = path1.parent.name if not path1.is_dir() else path1.name
    init_time = pd.Timestamp(time_str)
    init_ds = zarr_ds.sel(t=[init_time], method="nearest")

    if img_idx == 0:
        # Broadcast the initial state so the dimensions match for step 0
        dummy_ds = xr.open_dataset(join(forecast_dir, files[0]))
        if "ensemble_member_label" in dummy_ds.dims:
            init_ds_ens = init_ds.expand_dims(
                ensemble_member_label=dummy_ds.ensemble_member_label
            )
        else:
            init_ds_ens = init_ds

        plot_ensemble_diff_forecast(
            init_ds_ens, init_ds, init_ds, forecast_dir, channels, img_idx
        )
        gc.collect()
        return

    file = join(forecast_dir, file)
    ds = xr.open_dataset(file)
    true_ds = zarr_ds.sel(t=ds.t, method="nearest")

    plot_ensemble_diff_forecast(ds, true_ds, init_ds, forecast_dir, channels, img_idx)
    gc.collect()

f = partial(make_gif, forecast_dir, channels)
enum_files = [(-1, None)] + list(enumerate(files))

# Proceed with Parallel execution
result = Parallel(n_jobs=num_cpus)(
    delayed(f)(file_and_index) for file_and_index in enum_files
)


using 1 cpus
processing /glade/derecho/scratch/dkimpara/goes_10km_train/wx_inverted/forecasts/2022-07-01T00:25:06


: 

In [ ]:
path1 = pathlib.Path(forecast_dir)
time_str = path1.name

# Pointing to the new directory
gif_dir = join(path1.parent, "ensemble_diff_gif", f"gifs_{time_str}")

for channel in channels:
    res = subprocess.run(
        f"magick -delay {delay} -loop 0 {join(gif_dir, f'C{channel:02}/*.png')} {join(gif_dir, f'C{channel:02}.gif')}",
        shell=True,
        capture_output=True,
        encoding="utf-8",
    )
    print(f"created ensemble diff gif for C{channel:02}")
    shutil.rmtree(join(gif_dir, f"C{channel:02}"))
